# 08 - Mezzanine / 2nd Mortgage LGD Model

**Ranking-Driven Recovery Waterfall - Australian Bank-Style Portfolio Framework**

This notebook demonstrates a pragmatic LGD framework for subordinated real-estate lending.

| Step | Description |
|---|---|
| 1 | Generate synthetic mezz / 2nd mortgage defaults |
| 2 | Apply recovery waterfall (collateral -> senior debt -> residual to mezz) |
| 3 | Build base and macro-linked downturn LGD |
| 4 | Show ranking effect (mezz LGD > senior LGD) |
| 5 | Export EAD-weighted outputs and validation checks |


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(53)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")



## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** transparent proxy assumptions are used where observed internal mezz workout tapes are unavailable.
- **Cure / resolution treatment:** this module is waterfall-driven; no mortgage-style borrower cure model is forced into subordinated lien analysis.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status (standard policy):** this is a portfolio-project proxy baseline, **not** internally calibrated to real workout outcomes.
- **Output standard:** report `lgd_base`, `lgd_downturn`, `lgd_final`, EAD-weighted outputs, and base recovery-time metric (`time_to_recovery_base`).


## Objective
Build an interview-ready mezzanine/2nd mortgage LGD module with ranking-aware waterfall severity and weighted outputs.

## Drivers
- Total LVR
- Attachment point
- Subordination
- Collateral value decline and liquidity risk

## Logic
- Waterfall recovery (`collateral -> senior debt -> residual for mezz`) with base/downturn/final weighted LGD

## Downturn
- Macro-linked value/liquidity/rate stress widening the mezzanine ranking gap

## Validation
- Weighted scenario/segment/ranking outputs and validation checks

## Limitations
- Synthetic portfolio and proxy assumptions; future calibration requires internal workout data


## Why Mezz / 2nd Mortgage LGD Is Ranking-Driven

Mezzanine and second-mortgage recoveries are determined by **waterfall position**:

1. Realise collateral value net of stress and costs.
2. Repay senior debt first.
3. Residual value, if any, is available to mezz / 2nd lien.

This means subordinated ranking can produce high LGD even when collateral still has value.


## 1. Generate Synthetic Mezz / 2nd Mortgage Dataset


In [ ]:
n_cases = 260
rng = np.random.default_rng(53)

facility_types = ['mezzanine', 'second_mortgage']
facility_weights = [0.57, 0.43]

collateral_segments = ['residential_investment', 'cre_stabilised', 'residual_stock']
collateral_weights = [0.42, 0.38, 0.20]

segment_cfg = {
    'residential_investment': {'value_low': 0.8e6, 'value_high': 6.5e6, 'decline_mean': 0.10, 'liq_mean': 0.24},
    'cre_stabilised': {'value_low': 8.0e6, 'value_high': 120.0e6, 'decline_mean': 0.14, 'liq_mean': 0.33},
    'residual_stock': {'value_low': 6.0e6, 'value_high': 80.0e6, 'decline_mean': 0.19, 'liq_mean': 0.47},
}

attachment_means = {'mezzanine': 0.62, 'second_mortgage': 0.70}
subordination_means = {'mezzanine': 0.15, 'second_mortgage': 0.10}

rows = []
for i in range(1, n_cases + 1):
    facility_type = rng.choice(facility_types, p=facility_weights)
    collateral_segment = rng.choice(collateral_segments, p=collateral_weights)
    cfg = segment_cfg[collateral_segment]

    collateral_value = float(rng.uniform(cfg['value_low'], cfg['value_high']))

    attachment_point = float(np.clip(rng.normal(attachment_means[facility_type], 0.07), 0.40, 0.85))
    mezz_band = float(np.clip(rng.normal(subordination_means[facility_type], 0.03), 0.03, 0.24))
    total_lvr = float(np.clip(attachment_point + mezz_band, attachment_point + 0.02, 0.97))
    subordination = float(total_lvr - attachment_point)

    utilisation = float(np.clip(rng.normal(1.00, 0.03), 0.93, 1.08))
    senior_debt = collateral_value * attachment_point * utilisation
    mezz_ead = collateral_value * subordination * utilisation

    value_decline_base = float(np.clip(
        rng.normal(cfg['decline_mean'], 0.05) + 0.20 * max(total_lvr - 0.70, 0.0),
        0.04,
        0.45,
    ))
    market_liquidity_risk = float(np.clip(rng.normal(cfg['liq_mean'], 0.14), 0.05, 0.95))
    market_beta = float(np.clip(rng.normal(1.00, 0.18), 0.60, 1.60))

    sale_cost_rate_base = float(np.clip(0.045 + 0.040 * market_liquidity_risk + rng.normal(0, 0.006), 0.03, 0.14))
    time_to_recovery_months_base = float(np.clip(8 + 10 * market_liquidity_risk + 8 * subordination + rng.normal(0, 2.0), 4, 42))

    contract_rate_proxy = float(np.clip(rng.normal(0.112, 0.018), 0.070, 0.170))
    cost_of_funds_proxy = float(np.clip(rng.normal(0.066, 0.011), 0.040, 0.105))
    discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)

    rows.append({
        'loan_id': f'MEZ{i:04d}',
        'facility_type': facility_type,
        'collateral_segment': collateral_segment,
        'collateral_value': collateral_value,
        'senior_debt': senior_debt,
        'mezz_ead': mezz_ead,
        'total_lvr': total_lvr,
        'attachment_point': attachment_point,
        'subordination': subordination,
        'value_decline_base': value_decline_base,
        'market_liquidity_risk': market_liquidity_risk,
        'market_beta': market_beta,
        'sale_cost_rate_base': sale_cost_rate_base,
        'time_to_recovery_months_base': time_to_recovery_months_base,
        'contract_rate_proxy': contract_rate_proxy,
        'cost_of_funds_proxy': cost_of_funds_proxy,
        'discount_rate': discount_rate,
    })

mezz = pd.DataFrame(rows)

print('Mezz / 2nd mortgage cases:', mezz.shape)
display(mezz['facility_type'].value_counts())
display(mezz['collateral_segment'].value_counts())
mezz.head()



## 2. Recovery Waterfall: Collateral -> Senior Debt -> Residual for Mezz


In [ ]:
def run_mezz_waterfall(df, unemployment_shock=0.0, rate_shock=0.0, market_value_shock=0.0, liquidity_shock=0.0):
    x = df.copy()

    decline_rate = (
        x['value_decline_base']
        + market_value_shock * (0.75 + 0.45 * x['market_beta'])
        + unemployment_shock * (0.55 + 0.35 * x['market_liquidity_risk'])
        + rate_shock * (0.80 + 0.45 * (x['total_lvr'] - 0.65).clip(0, 0.35))
    ).clip(0.05, 0.80)

    sale_cost_rate = (
        x['sale_cost_rate_base']
        + 0.03 * liquidity_shock
        + 0.01 * x['market_liquidity_risk']
    ).clip(0.03, 0.20)

    time_to_recovery = (
        x['time_to_recovery_months_base']
        * (1 + 0.90 * liquidity_shock + 0.25 * x['market_liquidity_risk'] + 0.08 * (rate_shock / 0.01))
    ).clip(4, 60)

    discount_rate = (x['discount_rate'] + rate_shock).clip(0.04, 0.30)

    gross_collateral_after_decline = x['collateral_value'] * (1 - decline_rate)
    net_collateral = gross_collateral_after_decline * (1 - sale_cost_rate)

    senior_recovery_gross = np.minimum(net_collateral, x['senior_debt'])
    residual_for_mezz = (net_collateral - x['senior_debt']).clip(lower=0.0)

    discount_factor = (1 + discount_rate) ** (time_to_recovery / 12.0)
    senior_recovery_pv = senior_recovery_gross / discount_factor
    mezz_recovery_pv = residual_for_mezz / discount_factor

    senior_recovery_ratio = (senior_recovery_pv / x['senior_debt']).clip(0.0, 1.0)
    mezz_recovery_ratio = (mezz_recovery_pv / x['mezz_ead']).clip(0.0, 1.0)

    senior_lgd = (1.0 - senior_recovery_ratio).clip(0.0, 1.0)
    mezz_lgd = (1.0 - mezz_recovery_ratio).clip(0.0, 1.0)

    return pd.DataFrame({
        'decline_rate': decline_rate,
        'sale_cost_rate': sale_cost_rate,
        'time_to_recovery_months': time_to_recovery,
        'discount_rate': discount_rate,
        'net_collateral': net_collateral,
        'residual_for_mezz': residual_for_mezz,
        'senior_lgd': senior_lgd,
        'mezz_lgd': mezz_lgd,
    })


def weighted_senior_lgd(df, col):
    x = df[[col, 'senior_debt']].rename(columns={'senior_debt': 'ead'}).copy()
    return exposure_weighted_average(x, col, 'ead')


base_waterfall = run_mezz_waterfall(mezz)
for c in base_waterfall.columns:
    mezz[f'{c}_base'] = base_waterfall[c]

mezz['lgd_base'] = mezz['mezz_lgd_base']
mezz['ranking_gap_base'] = mezz['lgd_base'] - mezz['senior_lgd_base']

print(f"EAD-weighted base LGD (mezz): {exposure_weighted_average(mezz, 'lgd_base', 'mezz_ead'):.2%}")
print(f"EAD-weighted base LGD (senior): {weighted_senior_lgd(mezz, 'senior_lgd_base'):.2%}")
print(f"Base ranking gap (mezz - senior): {exposure_weighted_average(mezz, 'ranking_gap_base', 'mezz_ead'):.2%}")
display(mezz[['lgd_base', 'senior_lgd_base', 'ranking_gap_base', 'total_lvr', 'attachment_point', 'subordination']].describe().round(4))



## 3. Macro-Linked Downturn Scenarios and Final Overlay

Downturn scenarios stress value decline, rates, and liquidity timing.  
A small explicit final overlay is then added so cross-product comparison uses a consistent `lgd_final` metric for subordinated ranking risk.


In [ ]:
scenario_specs = [
    {'scenario': 'base', 'unemployment_shock': 0.000, 'rate_shock': 0.000, 'market_value_shock': 0.000, 'liquidity_shock': 0.000},
    {'scenario': 'value_stress', 'unemployment_shock': 0.010, 'rate_shock': 0.006, 'market_value_shock': 0.060, 'liquidity_shock': 0.060},
    {'scenario': 'liquidity_delay_stress', 'unemployment_shock': 0.015, 'rate_shock': 0.010, 'market_value_shock': 0.080, 'liquidity_shock': 0.140},
    {'scenario': 'combined_downturn', 'unemployment_shock': 0.025, 'rate_shock': 0.018, 'market_value_shock': 0.120, 'liquidity_shock': 0.220},
]

scenario_rows = []
for s in scenario_specs:
    wf = run_mezz_waterfall(
        mezz,
        unemployment_shock=s['unemployment_shock'],
        rate_shock=s['rate_shock'],
        market_value_shock=s['market_value_shock'],
        liquidity_shock=s['liquidity_shock'],
    )
    name = s['scenario']

    mezz[f'lgd_{name}'] = wf['mezz_lgd']
    mezz[f'senior_lgd_{name}'] = wf['senior_lgd']
    mezz[f'net_collateral_{name}'] = wf['net_collateral']
    mezz[f'residual_for_mezz_{name}'] = wf['residual_for_mezz']
    mezz[f'time_to_recovery_{name}'] = wf['time_to_recovery_months']
    mezz[f'ranking_gap_{name}'] = wf['mezz_lgd'] - wf['senior_lgd']

    overlay_s = (
        0.015
        + 0.050 * mezz['subordination']
        + 0.030 * (mezz['total_lvr'] - 0.75).clip(0.0, 0.25)
        + 0.020 * mezz['market_liquidity_risk']
    ).clip(0.015, 0.120)
    mezz[f'final_overlay_addon_{name}'] = overlay_s
    mezz[f'lgd_final_{name}'] = (wf['mezz_lgd'] + overlay_s).clip(0.0, 1.0)

    weighted_mezz = exposure_weighted_average(mezz, f'lgd_{name}', 'mezz_ead')
    scenario_rows.append({
        'scenario': name,
        'ead_weighted_lgd_base': weighted_mezz,
        'ead_weighted_lgd_downturn': weighted_mezz,
        'ead_weighted_lgd_mezz': weighted_mezz,
        'ead_weighted_lgd_senior': weighted_senior_lgd(mezz, f'senior_lgd_{name}'),
        'ead_weighted_lgd_final': exposure_weighted_average(mezz, f'lgd_final_{name}', 'mezz_ead'),
        'ead_weighted_ranking_gap': exposure_weighted_average(mezz, f'ranking_gap_{name}', 'mezz_ead'),
        'mean_time_to_recovery_months': mezz[f'time_to_recovery_{name}'].mean(),
        'mean_residual_coverage': np.average(
            (mezz[f'residual_for_mezz_{name}'] / mezz['mezz_ead']).clip(0.0, 1.0),
            weights=mezz['mezz_ead'],
        ),
    })

scenario_summary = pd.DataFrame(scenario_rows)
scenario_summary['mean_recovery_time_months'] = scenario_summary['mean_time_to_recovery_months']
mezz['lgd_downturn'] = mezz['lgd_combined_downturn']
mezz['senior_lgd_downturn'] = mezz['senior_lgd_combined_downturn']
mezz['ranking_gap_downturn'] = mezz['ranking_gap_combined_downturn']
mezz['final_overlay_addon'] = mezz['final_overlay_addon_combined_downturn']
mezz['lgd_final'] = mezz['lgd_final_combined_downturn']

display(scenario_summary.round(4))





## 4. Ranking Evidence: Subordination Drives Higher LGD


In [ ]:
ranking_summary = pd.DataFrame([
    {
        'measure': 'base',
        'ead_weighted_lgd_senior': weighted_senior_lgd(mezz, 'senior_lgd_base'),
        'ead_weighted_lgd_mezz': exposure_weighted_average(mezz, 'lgd_base', 'mezz_ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(mezz, 'lgd_final_base', 'mezz_ead'),
        'ead_weighted_ranking_gap': exposure_weighted_average(mezz, 'ranking_gap_base', 'mezz_ead'),
    },
    {
        'measure': 'downturn',
        'ead_weighted_lgd_senior': weighted_senior_lgd(mezz, 'senior_lgd_downturn'),
        'ead_weighted_lgd_mezz': exposure_weighted_average(mezz, 'lgd_downturn', 'mezz_ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(mezz, 'lgd_final', 'mezz_ead'),
        'ead_weighted_ranking_gap': exposure_weighted_average(mezz, 'ranking_gap_downturn', 'mezz_ead'),
    },
])

display(ranking_summary.round(4))

ranking_plot = scenario_summary[['scenario', 'ead_weighted_lgd_senior', 'ead_weighted_lgd_mezz', 'ead_weighted_lgd_final']].melt(
    id_vars='scenario',
    value_vars=['ead_weighted_lgd_senior', 'ead_weighted_lgd_mezz', 'ead_weighted_lgd_final'],
    var_name='rank_type',
    value_name='ead_weighted_lgd',
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=ranking_plot, x='scenario', y='ead_weighted_lgd', hue='rank_type', ax=ax)
ax.set_title('Ranking Effect: Senior vs Mezz vs Final EAD-Weighted LGD by Scenario')
ax.set_xlabel('Scenario')
ax.set_ylabel('EAD-Weighted LGD')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mezz_ranking_lgd_by_scenario.png'), dpi=140, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mezz_second_mortgage_weighted_lgd_comparison.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
    print('Plot display skipped (set LGD_NOTEBOOK_SHOW_PLOTS=1 to show).')



## 5. Segment Outputs, Validation Checks, and Exports


In [ ]:
mezz['total_lvr_bucket'] = pd.cut(
    mezz['total_lvr'],
    bins=[0.0, 0.70, 0.80, 0.90, 1.01],
    labels=['<=70%', '70-80%', '80-90%', '90%+'],
    right=False,
)

segment_rows = []
for (facility_type, collateral_segment, lvr_bucket), grp in mezz.groupby(
    ['facility_type', 'collateral_segment', 'total_lvr_bucket'],
    observed=True,
):
    segment_rows.append({
        'facility_type': facility_type,
        'collateral_segment': collateral_segment,
        'total_lvr_bucket': str(lvr_bucket),
        'loan_count': len(grp),
        'total_mezz_ead': grp['mezz_ead'].sum(),
        'ead_weighted_lgd_base': exposure_weighted_average(grp, 'lgd_base', 'mezz_ead'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(grp, 'lgd_downturn', 'mezz_ead'),
        'ead_weighted_lgd_final': exposure_weighted_average(grp, 'lgd_final', 'mezz_ead'),
        'mean_attachment_point': grp['attachment_point'].mean(),
        'mean_subordination': grp['subordination'].mean(),
        'mean_value_decline_base': grp['value_decline_base'].mean(),
    })

segment_summary = pd.DataFrame(segment_rows).sort_values(
    ['ead_weighted_lgd_final', 'total_mezz_ead'],
    ascending=[False, False],
).reset_index(drop=True)

display(segment_summary.head(20).round(4))

waterfall_snapshot = mezz[
    [
        'loan_id',
        'facility_type',
        'collateral_segment',
        'collateral_value',
        'senior_debt',
        'mezz_ead',
        'total_lvr',
        'attachment_point',
        'subordination',
        'net_collateral_base',
        'residual_for_mezz_base',
        'lgd_base',
        'lgd_downturn',
        'lgd_final',
    ]
].copy()

validation_checks = pd.DataFrame([
    {'check_name': 'lgd_base_between_0_and_1', 'passed': bool(mezz['lgd_base'].between(0, 1).all())},
    {'check_name': 'lgd_downturn_between_0_and_1', 'passed': bool(mezz['lgd_downturn'].between(0, 1).all())},
    {'check_name': 'lgd_final_between_0_and_1', 'passed': bool(mezz['lgd_final'].between(0, 1).all())},
    {'check_name': 'downturn_not_below_base', 'passed': bool((mezz['lgd_downturn'] >= mezz['lgd_base']).all())},
    {'check_name': 'final_not_below_downturn', 'passed': bool((mezz['lgd_final'] >= mezz['lgd_downturn']).all())},
    {
        'check_name': 'ranking_gap_positive_weighted_base',
        'passed': bool(exposure_weighted_average(mezz, 'ranking_gap_base', 'mezz_ead') > 0),
    },
    {
        'check_name': 'residual_mezz_coverage_lower_in_downturn',
        'passed': bool(mezz['residual_for_mezz_combined_downturn'].mean() <= mezz['residual_for_mezz_base'].mean()),
    },
])

display(validation_checks)

mezz.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mezz_second_mortgage_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mezz_second_mortgage_segment_summary.csv'), index=False)
scenario_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mezz_second_mortgage_scenario_summary.csv'), index=False)
ranking_summary.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mezz_second_mortgage_ranking_summary.csv'), index=False)
waterfall_snapshot.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mezz_second_mortgage_waterfall_snapshot.csv'), index=False)
validation_checks.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mezz_second_mortgage_validation_checks.csv'), index=False)

print('Saved mezz / 2nd mortgage outputs:')
print('- mezz_second_mortgage_loan_level_output.csv')
print('- mezz_second_mortgage_segment_summary.csv')
print('- mezz_second_mortgage_scenario_summary.csv')
print('- mezz_second_mortgage_ranking_summary.csv')
print('- mezz_second_mortgage_waterfall_snapshot.csv')
print('- mezz_second_mortgage_validation_checks.csv')



## Assumptions and Limitations

- Mezz / 2nd mortgage data is synthetic and for demonstration only.
- Waterfall mechanics are transparent proxies of legal/economic priority, not a substitute for transaction-specific intercreditor terms.
- Downturn logic is linked to macro drivers (value, unemployment, rates, liquidity), but coefficients are proxy assumptions pending internal calibration.
- Ranking comparison is the key message: subordinated debt depends on residual collateral value after senior repayment.


---

## APS 113 Calibration Layer — Mezzanine / Second Mortgage

> **SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY**
>
> This section adds a full APS 113-aligned calibration loop on top of the
> proxy baseline above. Workout data is synthetically generated (2014-2024,
> 10-year window) to demonstrate methodology; real production calibration
> requires an internal workout tape.

### Calibration Pipeline (APS 113 s.32-68)

| Step | Function | APS 113 |
|------|----------|---------|
| 1 | Load/generate historical workout data | s.44, Att A |
| 2 | `compute_realised_lgd()` — LIP costs, cure leg | s.32-34 |
| 3 | `classify_economic_regime()` + `assign_regime_to_workouts()` | s.43, s.46 |
| 4 | `segment_lgd()` — product-specific segment keys | s.52 |
| 5 | `compute_long_run_lgd()` — vintage-EWA method | s.43-44 |
| 6 | `compare_model_vs_actual()` — proxy vs realised | s.60-62 |
| 7 | `apply_downturn_overlay()` + `apply_correlation_adjustment()` | s.46-50, s.55-57 |
| 8 | `MoCRegister` + `apply_moc()` — 5 APS 113 s.65 sources | s.63-65 |
| 9 | `apply_regulatory_floor()` — 40% per APS 113 s.58 — junior ranking | s.58 |
| 10 | Export 9 CSV outputs | — |
| 11 | `run_full_validation_suite()` — Gini, HL, PSI, OOT | s.66-68 |

**Correct APS 113 order:** LR-LGD → downturn overlay → correlation adj →
MoC → floor (MoC applied to downturn LGD, not long-run LGD, per s.63).


In [ ]:
# APS 113 Calibration Layer — imports and configuration
import os, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))

from src.calibration_utils import (
    compute_realised_lgd,
    segment_lgd,
    compute_long_run_lgd,
    compare_model_vs_actual,
    classify_economic_regime,
    assign_regime_to_workouts,
    apply_downturn_overlay,
    apply_correlation_adjustment,
    build_lgd_pd_annual_series,
    estimate_lgd_pd_correlation,
    MoCRegister,
    apply_moc,
    apply_regulatory_floor,
    run_full_validation_suite,
    generate_compliance_map,
    export_compliance_map,
    CALIBRATION_STEP_ORDER,
)
from src.generators import GENERATOR_MAP, generate_all_historical_workouts

PRODUCT = "mezz_second_mortgage"
SEGMENT_KEYS = ['seniority', 'attachment_point_band']
MODEL_LGD_COL = "lgd_final"

HISTORY_DIR = Path('..') / 'data' / 'generated' / 'historical'
OUTPUTS_DIR = Path('..') / 'outputs' / 'tables'
UPSTREAM_PARQUET = Path('..') / 'data' / 'exports' / 'macro_regime_flags.parquet'

HISTORY_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Product: {PRODUCT}")
print(f"Segment keys: {SEGMENT_KEYS}")
print(f"APS 113 calibration pipeline — step order:")
for step, fn, ref in CALIBRATION_STEP_ORDER:
    print(f"  Step {step:>2}: {fn:<35} {ref}")


In [ ]:
# ── STEP 1: Load or generate historical workout data (APS 113 s.44, Att A) ─
parquet_path = HISTORY_DIR / f"{PRODUCT}_workouts.parquet"

if parquet_path.exists():
    cal_loans = pd.read_parquet(parquet_path)
    # cashflows stored alongside or re-generated
    try:
        cal_cashflows = pd.read_parquet(
            HISTORY_DIR / f"{PRODUCT}_cashflows.parquet"
        )
    except FileNotFoundError:
        cal_cashflows = None
    print(f"Loaded {len(cal_loans):,} historical workout loans from Parquet")
else:
    print(f"Parquet not found — generating synthetic workout data for {PRODUCT}")
    results = generate_all_historical_workouts(
        seed=42, output_dir=HISTORY_DIR, write_parquet=True, products=[PRODUCT]
    )
    cal_loans = results[PRODUCT]["loans"]
    cal_cashflows = results[PRODUCT].get("cashflows")
    print(f"Generated {len(cal_loans):,} synthetic workout loans (2014-2024)")

print(f"Date range: {cal_loans['default_date'].min()} to {cal_loans['default_date'].max()}")
print(f"Columns: {list(cal_loans.columns)}")

# ── STEP 2: Compute realised LGD (APS 113 s.32-34) ─────────────────────────
# LIP costs (Loss Identification Period, ~90 days) auto-detected if cashflows available
lgd_df = compute_realised_lgd(
    loans=cal_loans,
    cashflows=cal_cashflows,
    ead_col="ead_at_default",
    recovery_col="recovery_amount",
    cost_col="direct_costs",
    lip_window_days=90,
)
print(f"\nStep 2: Realised LGD computed")
print(f"  EAD-weighted realised LGD: {(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum():.2%}")
display(lgd_df[['realised_lgd', 'ead_at_default']].describe().round(4))

# ── STEP 3: Economic regime classification (APS 113 s.43, s.46-50) ─────────
upstream_path = str(UPSTREAM_PARQUET) if UPSTREAM_PARQUET.exists() else None
regime_df = classify_economic_regime(
    upstream_parquet_path=upstream_path,
    method="upstream_first",
)
print(f"\nStep 3: Economic regimes classified")
print(f"  Data source: {regime_df['data_source'].iloc[0]}")
display(regime_df[['year', 'regime', 'is_downturn_period', 'data_source']].head(12))

lgd_df = assign_regime_to_workouts(lgd_df, regime_df)
downturn_pct = lgd_df.get('is_downturn_period', pd.Series([False])).mean()
print(f"  Downturn observations: {downturn_pct:.1%} of portfolio")

# ── STEP 4: Segment by product-specific keys (APS 113 s.52) ────────────────
segmented_df = segment_lgd(lgd_df, SEGMENT_KEYS)
low_count = segmented_df[segmented_df.get('segment_flag', '') == 'low_count'] if 'segment_flag' in segmented_df.columns else pd.DataFrame()
print(f"\nStep 4: Segmentation complete")
print(f"  Segments: {segmented_df.groupby(SEGMENT_KEYS, observed=True).ngroups}")
if not low_count.empty:
    print(f"  Low-count segments flagged (<20 obs): {len(low_count)}")

# ── STEP 5: Long-run LGD — vintage-EWA (APS 113 s.43-44) ─────────────────
lr_lgd_df = compute_long_run_lgd(
    segmented_df,
    segment_keys=SEGMENT_KEYS,
    method="vintage_ewa",
)
print(f"\nStep 5: Long-run LGD by segment (vintage-EWA)")
display(lr_lgd_df.round(4))
lr_lgd_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_long_run_lgd_by_segment.csv", index=False)

# ── STEP 6: Compare model vs actual (APS 113 s.60-62) ──────────────────────
# 'model_lgd' here = proxy model LGD from the section above (lgd_final if present)
if MODEL_LGD_COL in cal_loans.columns:
    compare_input = lgd_df.merge(
        cal_loans[['loan_id', MODEL_LGD_COL] if 'loan_id' in cal_loans.columns else [MODEL_LGD_COL]],
        left_index=True, right_index=True, how='left',
    ) if MODEL_LGD_COL not in lgd_df.columns else lgd_df
else:
    compare_input = lgd_df.copy()
    compare_input['model_lgd'] = lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else 0.25

model_col_to_use = MODEL_LGD_COL if MODEL_LGD_COL in compare_input.columns else 'model_lgd'
mva_df = compare_model_vs_actual(
    compare_input,
    model_lgd_col=model_col_to_use,
    segment_keys=SEGMENT_KEYS,
)
print(f"\nStep 6: Model vs actual comparison")
display(mva_df.round(4))
mva_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_model_vs_actual_comparison.csv", index=False)

# ── STEP 7: Downturn overlay + Frye-Jacobs correlation adj (s.46-50, s.55-57)
# Reuses apply_downturn_overlay from existing lgd_calculation.py
dt_lgd = apply_downturn_overlay(segmented_df, product=PRODUCT)
print(f"\nStep 7: Downturn overlay applied")
if 'lgd_downturn' in dt_lgd.columns:
    ewa_dt = (dt_lgd['lgd_downturn'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    ewa_lr = (dt_lgd['realised_lgd'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    print(f"  EWA Long-run LGD:  {ewa_lr:.2%}")
    print(f"  EWA Downturn LGD:  {ewa_dt:.2%}")
    downturn_col = 'lgd_downturn'
else:
    dt_lgd['lgd_downturn'] = dt_lgd['realised_lgd'] * 1.15  # fallback scalar
    downturn_col = 'lgd_downturn'

# Frye-Jacobs correlation adjustment (APS 113 s.55-57)
try:
    lgd_ts, pd_ts = build_lgd_pd_annual_series(dt_lgd)
    macro_for_corr = regime_df.rename(columns={'gdp_growth': 'gdp_growth_yoy'})
    corr_result = estimate_lgd_pd_correlation(lgd_ts, pd_ts, macro_for_corr)
    dt_lgd['lgd_downturn_corr_adj'] = apply_correlation_adjustment(
        dt_lgd[downturn_col], corr_result['rho'], corr_result['macro_shock_std']
    )
    downturn_col = 'lgd_downturn_corr_adj'
    print(f"  Frye-Jacobs rho={corr_result['rho']:.3f}, adj factor={corr_result['lgd_dt_adjustment_factor']:.3f}")
except Exception as exc:
    print(f"  Frye-Jacobs skipped: {exc}")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_downturn_lgd_by_segment.csv", index=False)

# ── STEP 8: MoC register + apply MoC (AFTER downturn — APS 113 s.63-65) ───
# Determine regime data source for MoC model_approximation component
data_src = regime_df['data_source'].iloc[0] if 'data_source' in regime_df.columns else 'synthetic'
n_downturn_yrs = int(regime_df['is_downturn_period'].sum()) if 'is_downturn_period' in regime_df.columns else 0

psi_approx = 0.05  # placeholder — full PSI computed in Step 11
bias_approx = float(mva_df['bias'].abs().mean()) if 'bias' in mva_df.columns else 0.02

moc_register = MoCRegister(product=PRODUCT, regime_data_source=data_src)
moc_df = moc_register.build_moc_register(
    segment_df=segmented_df,
    segment_keys=SEGMENT_KEYS,
    n_downturn_vintages=n_downturn_yrs,
    psi_value=psi_approx,
    backtesting_bias=bias_approx,
)
print(f"\nStep 8: MoC register built")
display(moc_df.round(4))
moc_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_moc_register.csv", index=False)

lgd_with_moc = apply_moc(dt_lgd[downturn_col], moc_df, segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None)
dt_lgd['lgd_with_moc'] = lgd_with_moc
moc_ewa = (lgd_with_moc * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
print(f"  EWA LGD after MoC: {moc_ewa:.2%}")

# ── STEP 9: Regulatory floors (APS 113 s.58) ──────────────────────────────
dt_lgd['lgd_final_calibrated'] = apply_regulatory_floor(dt_lgd['lgd_with_moc'], product=PRODUCT)
final_ewa = (dt_lgd['lgd_final_calibrated'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
floor_binding_pct = (dt_lgd['lgd_final_calibrated'] > dt_lgd['lgd_with_moc']).mean()
print(f"\nStep 9: Regulatory floor applied")
print(f"  EWA Final Calibrated LGD: {final_ewa:.2%}")
print(f"  Floor binding for: {floor_binding_pct:.1%} of loans")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_final_calibrated_lgd.csv", index=False)

# ── STEP 10: Export all outputs ────────────────────────────────────────────
# Already exported: long_run_lgd_by_segment, model_vs_actual, downturn_lgd, moc_register, final_calibrated_lgd
# Export remaining:
lgd_df[['realised_lgd', 'ead_at_default']].assign(product=PRODUCT).to_csv(
    OUTPUTS_DIR / f"{PRODUCT}_historical_workouts.csv", index=False
)
regime_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_regime_classification.csv", index=False)

# Calibration adjustments summary
cal_adj_summary = pd.DataFrame({
    'product': [PRODUCT],
    'ewa_realised_lgd': [(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum()],
    'ewa_long_run_lgd': [lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None],
    'ewa_downturn_lgd': [(dt_lgd.get('lgd_downturn', dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_with_moc': [(dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_final': [final_ewa],
    'floor_binding_pct': [floor_binding_pct],
    'regime_data_source': [data_src],
    'n_loans': [len(lgd_df)],
    'calibration_date': [pd.Timestamp.today().date()],
})
cal_adj_summary.to_csv(OUTPUTS_DIR / f"{PRODUCT}_calibration_adjustments.csv", index=False)
print(f"\nStep 10: All outputs exported to {OUTPUTS_DIR}")

# ── STEP 11: Full validation suite (APS 113 s.66-68) ──────────────────────
print(f"\nStep 11: Running full validation suite...")
try:
    val_results = run_full_validation_suite(
        loans=dt_lgd,
        predicted_col='lgd_final_calibrated',
        actual_col='realised_lgd',
        segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None,
        date_col='default_date' if 'default_date' in dt_lgd.columns else None,
        product=PRODUCT,
    )
    print(f"  Gini: {val_results.get('gini', 'n/a')}")
    print(f"  Calibration ratio: {val_results.get('calibration_ratio', 'n/a')}")
    print(f"  PSI: {val_results.get('psi', 'n/a')}")
    if 'summary_table' in val_results:
        val_results['summary_table'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_validation_report.csv", index=False
        )
    if 'backtest_results' in val_results:
        val_results['backtest_results'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_backtest_results.csv", index=False
        )
except Exception as exc:
    print(f"  Validation suite error (non-fatal): {exc}")


In [ ]:
# ── Calibration summary waterfall ──────────────────────────────────────────
import matplotlib.pyplot as plt

try:
    stages = {
        'Realised LGD\n(2014-2024)': (lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum(),
        'Long-run LGD\n(vintage-EWA)': lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None,
        'Downturn LGD': (dt_lgd.get(downturn_col, dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        '+ MoC': (dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        'Final\n(Floor Applied)': final_ewa,
    }
    labels = [k for k, v in stages.items() if v is not None]
    values = [v for v in stages.values() if v is not None]
    colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#8e44ad'][:len(labels)]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_ylabel('EAD-Weighted LGD')
    ax.set_title(f'APS 113 Calibration Waterfall — Mezzanine / Second Mortgage')
    ax.set_ylim(0, max(values) * 1.35)
    ax.axhline(values[-1], color='black', ls=':', lw=1, label=f'Final: {values[-1]:.1%}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        Path('..') / 'outputs' / 'figures' / f'mezz_second_mortgage_calibration_waterfall.png',
        dpi=150, bbox_inches='tight',
    )
    plt.show()
except Exception as exc:
    print(f"Waterfall chart error (non-fatal): {exc}")

# ── APS 113 compliance snapshot ─────────────────────────────────────────────
compliance_df = generate_compliance_map(
    calibration_results={PRODUCT: {'long_run_lgd_by_segment': True, 'calibration_steps': True}},
    moc_registers={PRODUCT: moc_df},
    regime_data_source=data_src,
    products=[PRODUCT],
)
print("\n=== APS 113 Compliance Snapshot ===")
display(compliance_df[['section_ref', 'requirement', 'status', 'reviewer_note']].set_index('section_ref'))
export_compliance_map(compliance_df, OUTPUTS_DIR / f"mezz_second_mortgage_aps113_compliance.csv")

# Final summary
print("\n=== Calibration Summary ===")
display(cal_adj_summary.round(4))
print(f"\nAll calibration outputs in: {OUTPUTS_DIR}")
print(f"SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY")
